# Hanoi Traffic Congestion Prediction

## Project Overview

Predict congestion ratio (travel time / free-flow time) for 40 routes in Hanoi using TomTom API data. Features include time of day, weather, route characteristics, and recent traffic history.

**Key results:**
- XGBoost R²: 0.98
- XGBoost R²: 0.75 (without recent history)
- MAE: 0.019 (approx 1 minute error for a 60-minute trip)

**Data:** 10,000+ observations, 40 routes, 11 days, 30-min intervals

## 1. Setup

In [14]:
!pip install optuna

import joblib
import optuna
import xgboost as xgb
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder
from sqlalchemy import create_engine
from google.colab import userdata, files

In [15]:
DB_URL = userdata.get('SUPABASE_DB_URL')

engine = create_engine(DB_URL)

query = """
SELECT
    t.route_id,
    t.congestion_ratio,
    t.observed_time,
    t.travel_time_seconds,
    t.no_traffic_time_seconds,
    EXTRACT(HOUR FROM t.observed_time) as hour,
    EXTRACT(MINUTE FROM t.observed_time) as minute,
    EXTRACT(DOW FROM t.observed_time) as day_of_week,
    EXTRACT(DAY FROM t.observed_time) as date,
    rg.length_meters,
    rg.straightness_ratio,
    w.temperature_celsius,
    w.weather_condition,
    CASE WHEN h.holiday_date IS NOT NULL THEN 1 ELSE 0 END as is_holiday,
    CASE WHEN t.is_rush_hour IS TRUE THEN 1 ELSE 0 END as is_rush_hour
FROM traffic_observations t
JOIN routes r ON t.route_id = r.route_id
LEFT JOIN route_geometries rg ON r.route_id = rg.route_id
INNER JOIN weather_data w ON DATE(t.observed_time) = w.observation_date
    AND EXTRACT(HOUR FROM t.observed_time) = w.hour
LEFT JOIN holidays h ON DATE(t.observed_time) = h.holiday_date
WHERE t.congestion_ratio IS NOT NULL
"""

df = pd.read_sql(query, engine).copy()

seed = 85

## 2. Feature Engineering

### 2.1 Time Features

In [16]:
# Cyclical encoding for hour (24-hour cycle)
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

# Cyclical encoding for minute (60-minute cycle)
df['minute_sin'] = np.sin(2 * np.pi * df['minute'] / 60)
df['minute_cos'] = np.cos(2 * np.pi * df['minute'] / 60)

# Cyclical encoding for day of week (7-day cycle)
df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

# Linear time feature (minutes since 6 AM)
df['minutes_elapsed'] = (df['hour'] - 6) * 60 + df['minute']
df['minutes_elapsed'] = df['minutes_elapsed'].clip(0, 780)
df['minutes_elapsed_norm'] = df['minutes_elapsed'] / 780

### 2.2 Weather Features

In [17]:
df['temperature_norm'] = (
    df['temperature_celsius'] - df['temperature_celsius'].min()) / (
    df['temperature_celsius'].max() - df['temperature_celsius'].min()
)

df['weather_encoded'] = LabelEncoder().fit_transform(df['weather_condition'])

### 2.3 Interaction Features

In [18]:
df['rush_hour_route'] = df['is_rush_hour'] * df['route_id']
df['hour_sin_length'] = df['hour_sin'] * df['length_meters']
df['hour_sin_dow'] = df['hour_sin'] * df['dow_sin']
df['hour_cos_dow'] = df['hour_cos'] * df['dow_cos']

### 2.4 Lag and Rolling Features

These features are computed within each (route, date) group to prevent cross-day leakage.

In [19]:
df = df.sort_values(['route_id', 'observed_time'])
congestion_by_route_and_date = df.groupby(['route_id', 'date'])['congestion_ratio']

df['congestion_lag_1'] = congestion_by_route_and_date.shift(1)
df['congestion_lag_2'] = congestion_by_route_and_date.shift(2)
df['congestion_rolling_mean_3'] = congestion_by_route_and_date.transform(
    lambda x: x.rolling(3, min_periods=1).mean()
)
df['congestion_rolling_std_3'] = congestion_by_route_and_date.transform(
    lambda x: x.rolling(3, min_periods=1).std().fillna(0)
)

# Drop rows without enough history
df = df.dropna(subset=['congestion_lag_2'])

### 2.5 Feature Selection

In [20]:
feature_columns = [
    # Time features (cyclical)
    'hour_sin', 'hour_cos',
    'minute_sin', 'minute_cos',
    'dow_sin', 'dow_cos',
    'minutes_elapsed',

    # Rush hour
    'is_rush_hour',

    # Weather
    'weather_encoded',
    'temperature_celsius',

    # Lag features
    'congestion_lag_1', 'congestion_lag_2',
    'congestion_rolling_mean_3',
    'congestion_rolling_std_3',

    # Route features
    'length_meters',
    'straightness_ratio',

    # Interaction features
    'rush_hour_route',
    'hour_sin_length',
    'hour_sin_dow',
    'hour_cos_dow',

    # Holiday
    'is_holiday'
]

# Target
target_column = 'congestion_ratio'

X = df[feature_columns]
y = df[target_column]

## 3. Model Training

### 3.1 XGBoost with Hyperparameter Tuning

In [21]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 1000),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
    }
    model = xgb.XGBRegressor(**params, random_state=seed)
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='r2').mean()
    return score

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

best_model = xgb.XGBRegressor(**study.best_params, random_state=seed)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)
print(f"MAE: {mean_absolute_error(y_test, y_pred):.3f}")
print(f"R2: {r2_score(y_test, y_pred):.3f}")
print(f"Best parameters: {study.best_params}")

[I 2026-05-28 01:17:32,628] A new study created in memory with name: no-name-e4355e76-df2f-4cdf-a4f8-d8401651eead
[I 2026-05-28 01:17:48,059] Trial 0 finished with value: 0.9532514992798021 and parameters: {'n_estimators': 689, 'max_depth': 5, 'learning_rate': 0.09251741303095223, 'subsample': 0.7932660619090249, 'colsample_bytree': 0.9945830276956407, 'reg_alpha': 6.010714847456243e-08, 'reg_lambda': 5.765035980666125e-06, 'min_child_weight': 2}. Best is trial 0 with value: 0.9532514992798021.
[I 2026-05-28 01:17:54,996] Trial 1 finished with value: 0.9576781956098737 and parameters: {'n_estimators': 860, 'max_depth': 4, 'learning_rate': 0.08831983551664283, 'subsample': 0.6540493256528576, 'colsample_bytree': 0.886012088216392, 'reg_alpha': 1.0879650996211374e-06, 'reg_lambda': 7.165174669481465e-07, 'min_child_weight': 3}. Best is trial 1 with value: 0.9576781956098737.
[I 2026-05-28 01:18:00,839] Trial 2 finished with value: 0.9361062236201199 and parameters: {'n_estimators': 429, 

MAE: 0.019
R2: 0.975
Best parameters: {'n_estimators': 655, 'max_depth': 4, 'learning_rate': 0.09144985185925458, 'subsample': 0.9327285987625586, 'colsample_bytree': 0.8609093932696518, 'reg_alpha': 3.4628403508537565e-07, 'reg_lambda': 0.05199051053840526, 'min_child_weight': 10}


Optuna performs 50 trials of Bayesian optimisation to find the best XGBoost parameters.

The tuned XGBoost achieves R² = 0.975 on the test set. MAE = 0.019 means prediction error is approximately 1.9% of the free-flow travel time.

In [22]:
importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=False)
print(importance.head(10))

                      feature  importance
12  congestion_rolling_mean_3    0.512057
10           congestion_lag_1    0.098535
11           congestion_lag_2    0.060395
13   congestion_rolling_std_3    0.043638
6             minutes_elapsed    0.036218
4                     dow_sin    0.035971
19               hour_cos_dow    0.034762
1                    hour_cos    0.033654
0                    hour_sin    0.032488
5                     dow_cos    0.021015


Rolling mean of congestion (past 90 minutes) is the strongest predictor, followed by both lag features (30 min and 60 min ago). This confirms that recent traffic history dominates short-term predictions.

### 3.2 Without Rolling or Lag Features (Baseline Comparison)

This shows how much predictive power comes from recent traffic history.

In [28]:
features_no_recent = [f for f in feature_columns \
                      if 'rolling' not in f and 'lag' not in f]
X_train_no_recent = X_train[features_no_recent]

model_no_recent = xgb.XGBRegressor(**study.best_params)
model_no_recent.fit(X_train_no_recent, y_train)
y_pred_no_recent = model_no_recent.predict(X_test[features_no_recent])

importance_no_recent = pd.DataFrame({
    'feature': X_train_no_recent.columns,
    'importance': model_no_recent.feature_importances_
}).sort_values('importance', ascending=False)

r2_no_recent = r2_score(y_test, y_pred_no_recent)
mae_no_recent = mean_absolute_error(y_test, y_pred_no_recent)
print(f"R² without rolling or lag features: {r2_no_recent:.4f}")
print(f"MAE without rolling or lag features: {mae_no_recent:.3f}")
print(importance_no_recent.head(10))

R² without rolling or lag features: 0.7505
MAE without rolling or lag features: 0.094
               feature  importance
7         is_rush_hour    0.207476
1             hour_cos    0.144518
11  straightness_ratio    0.116354
12     rush_hour_route    0.096150
5              dow_cos    0.088981
14        hour_sin_dow    0.055416
4              dow_sin    0.051549
0             hour_sin    0.046422
6      minutes_elapsed    0.030549
10       length_meters    0.030386


The model is saved locally from Colab and can be downloaded for use in the Streamlit dashboard.

In [26]:
joblib.dump(model_no_recent, 'traffic_model.pkl')
joblib.dump(features_no_recent, 'feature_columns.pkl')
metadata = {
    'feature_columns': features_no_recent,
    'target_column': 'congestion_ratio',
    'model_version': '2026-05-27',
    'train_date_range': ['2026-05-17', '2026-05-27'],
    'n_observations': 10638,
    'n_routes': 40,
    'r2_score': r2_no_recent,
    'mae': mae_no_recent
}
joblib.dump(metadata, 'model_metadata.pkl')

files.download('traffic_model.pkl')
files.download('feature_columns.pkl')
files.download('model_metadata.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 3.3 Time Series Cross-Validation

Using expanding window to prevent future data leakage.

In [25]:
tscv = TimeSeriesSplit(n_splits=5)
cv_scores = []

for train_idx, val_idx in tscv.split(X):
    X_train_cv, X_val_cv = X.iloc[train_idx], X.iloc[val_idx]
    y_train_cv, y_val_cv = y.iloc[train_idx], y.iloc[val_idx]

    model = xgb.XGBRegressor(**study.best_params)
    model.fit(X_train_cv, y_train_cv)
    y_pred_cv = model.predict(X_val_cv)
    cv_scores.append(r2_score(y_val_cv, y_pred_cv))

print(f"Time series CV R²: mean={np.mean(cv_scores):.4f}, std={np.std(cv_scores):.4f}")

Time series CV R²: mean=0.8319, std=0.2836


The high variance across folds (std = 0.2836) indicates that model performance depends on the specific time period. This is expected with only 11 days of data. With more data (14+ days), the standard deviation would decrease.


## 4. Data Stability Analysis

Traffic stability directly affects model performance. High stability means high R² is expected.

In [27]:
print(f"Congestion ratio variance: {df['congestion_ratio'].var():.4f}")
print(f"Congestion ratio range: {df['congestion_ratio'].min():.2f} - {df['congestion_ratio'].max():.2f}")
print(f"Congestion ratio std: {df['congestion_ratio'].std():.4f}")
print()

# Check variance across full day for each route
group_cols = ['route_id', 'date']
variance_by_day = df.groupby(group_cols)['congestion_ratio'].var()
print(f"Daily variance per route - mean: {variance_by_day.mean():.4f}")
print(f"Daily variance per route - std: {variance_by_day.std():.4f}")
print(f"Days with variance < 0.05: {(variance_by_day < 0.05).sum()} / {len(variance_by_day)}")
print()

# Check variance at same hour across different days
group_cols = ['route_id', 'hour']
variance_by_hour = df.groupby(group_cols)['congestion_ratio'].var()
print(f"Hourly variance across days - mean: {variance_by_hour.mean():.4f}")
print(f"Hours with variance < 0.05: {(variance_by_hour < 0.05).sum()} / {len(variance_by_hour)}")
print()

# Check how much congestion changes between consecutive observations
df = df.sort_values(['route_id', 'date', 'observed_time'])
df['congestion_change'] = df.groupby(['route_id', 'date'])['congestion_ratio'].diff().abs()
print(f"Mean absolute change between 30-min intervals: {df['congestion_change'].mean():.4f}")
print(f"Median absolute change: {df['congestion_change'].median():.4f}")
print(f"95th percentile change: {df['congestion_change'].quantile(0.95):.4f}")
print()

Congestion ratio variance: 0.1229
Congestion ratio range: 0.92 - 5.67
Congestion ratio std: 0.3506

Daily variance per route - mean: 0.0713
Daily variance per route - std: 0.1379
Days with variance < 0.05: 267 / 400

Hourly variance across days - mean: 0.0521
Hours with variance < 0.05: 451 / 600

Mean absolute change between 30-min intervals: 0.1131
Median absolute change: 0.0500
95th percentile change: 0.4400



These statistics confirm that traffic in this dataset is inherently stable. A simple "no change" baseline would achieve high accuracy. The model's R² = 0.98 reflects this stability, not necessarily sophisticated pattern discovery.

## Conclusion

**What works:** Short-term congestion prediction using recent traffic history is highly accurate (R² = 0.975, MAE = 0.019).

**Limitations:** Without recent history, accuracy drops significantly to R² = 0.751. The inherent stability of traffic data inflates the R².

**Future improvements:** Continue collecting more data, add night hours (7 PM to 6 AM), and expand to more routes.